In [1]:

import os
import glob
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pandas as pd
from collections import defaultdict


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_GPUS = torch.cuda.device_count()
print(f"Using {NUM_GPUS} GPU(s)")

SEQ_LEN = 3
IMG_SIZE = 192         
BATCH_SIZE = 8         
EPOCHS = 7
LR = 1e-4
HIDDEN_CH = 64
DROPOUT = 0.5

TRAIN_IMG_DIR = "/kaggle/input/pixel-play-26/Avenue_Corrupted-20251221T112159Z-3-001/Avenue_Corrupted/Dataset/training_videos"
TEST_IMG_DIR  = "/kaggle/input/pixel-play-26/Avenue_Corrupted-20251221T112159Z-3-001/Avenue_Corrupted/Dataset/testing_videos"

MODEL_PATH = "model_cnn_convlstm1.pth"
SUBMISSION_PATH = "submission1.csv"

# IMPORTANT:
# This must be the CSV that worked previously (correct row count)
TEMPLATE_CSV = "/kaggle/input/sample/submission-8.csv"

def extract_frame_number(path):
    return int(os.path.basename(path).split("_")[1].split(".")[0])

def preprocess_frame(path):
    img = cv2.imread(path) # , cv2.IMREAD_GRAYSCALE)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.bilateralFilter(img, 5, 75, 75)
    img = img.astype(np.float32) / 255.0
    return img

def unflip_if_needed(img):
    # Fix corrupted vertically flipped frames
    if img[:IMG_SIZE//2].mean() > img[IMG_SIZE//2:].mean():
        return np.flipud(img)
    return img

class AvenuePredictionDataset(Dataset):
    def __init__(self, root):
        self.samples = []

        for vid in sorted(os.listdir(root)):
            paths = sorted(glob.glob(os.path.join(root, vid, "*.jpg")))
            frames = [unflip_if_needed(preprocess_frame(p)) for p in paths]
            frames = np.stack(frames)

            for i in range(SEQ_LEN, len(frames) - 1):
                self.samples.append((frames, i))

    def __len__(self):
        return len(self.samples)


    def __getitem__(self, idx):
        frames, i = self.samples[idx]
    
        seq = frames[i-SEQ_LEN:i]            # (T, H, W)
    
        # compute frame differences
        diffs = seq[1:] - seq[:-1]            # (T-1, H, W)
        diffs = np.concatenate(
            [np.zeros_like(diffs[:1]), diffs], axis=0
        )                                     # (T, H, W)
# for rgb
        seq = seq.transpose(0, 3, 1, 2)      # (T, 3, H, W)
        diffs = diffs.transpose(0, 3, 1, 2)  # (T, 3, H, W)

        x = np.concatenate([seq, diffs], axis=1)  # (T, 6, H, W)
        y = frames[i].transpose(2, 0, 1)          # (3, H, W)

        #for grayscale
    
        # # stack frame + diff as channels
        # x = np.stack([seq, diffs], axis=1)    # (T, 2, H, W)
    
        # y = frames[i]                         # next frame
        return torch.tensor(x), torch.tensor(y)


        


class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden_ch):
        super().__init__()
        self.hidden_ch = hidden_ch

        self.conv = nn.Conv2d(
            in_ch + hidden_ch,
            4 * hidden_ch,
            kernel_size=3,
            padding=1
        )
        self.bn = nn.BatchNorm2d(4 * hidden_ch)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates = self.bn(self.conv(combined))

        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)

        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c


class CNNConvLSTMPredictor(nn.Module):
    def __init__(self):
        super().__init__()

        # CNN encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(6, 32, 3, padding=1), # change to 2 for grayscale
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout2d(DROPOUT),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.convlstm = ConvLSTMCell(64, HIDDEN_CH)

        # CNN decoder
        self.decoder = nn.Sequential(
            nn.Conv2d(HIDDEN_CH, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout2d(DROPOUT),

            nn.Conv2d(32, 3, 3, padding=1), # change to 32 1 for grayscale
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: [B, T, C, H, W]
        B, T, C, H, W = x.shape
    
        h = torch.zeros(B, HIDDEN_CH, H, W, device=x.device)
        c = torch.zeros_like(h)
    
        for t in range(T):
            feat = self.encoder(x[:, t])   # [B, C, H, W] → [B, 64, H, W]
            h, c = self.convlstm(feat, h, c)
    
        out = self.decoder(h)              # [B, 1, H, W]
        return out

def train():
    dataset = AvenuePredictionDataset(TRAIN_IMG_DIR)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=8,
        pin_memory=True
    )

    model = CNNConvLSTMPredictor().to(DEVICE)
    if NUM_GPUS > 1:
        model = nn.DataParallel(model)
    SAVE_EPOCHS = [1,2,3,4,5,6,7]

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for x, y in tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            pred = model(x)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} | Loss: {total_loss/len(loader):.6f}")

        if epoch+1 in SAVE_EPOCHS:
            state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
            torch.save(state, f"model_epoch{epoch+1}.pth")
            print(f"Saved model_epoch{epoch+1}.pth")



 

def generate_submission():
    # Load template (already has ALL required IDs)
    sample = pd.read_csv(TEMPLATE_CSV)
    sample["Predicted"] = 0.0

    model = CNNConvLSTMPredictor().to(DEVICE)
    model.load_state_dict(torch.load( "/kaggle/working/model_epoch5.pth",map_location=DEVICE))
    model.eval()

    score_map = {}
    all_scores = []

    for vid in tqdm(sorted(os.listdir(TEST_IMG_DIR)),desc = "Vids"):
        paths = sorted(glob.glob(os.path.join(TEST_IMG_DIR, vid, "*.jpg")))
        frames = [unflip_if_needed(preprocess_frame(p)) for p in paths]
        frames = np.stack(frames)
        frame_nums = [extract_frame_number(p) for p in paths]

        for i in range(SEQ_LEN, len(frames) - 1):
            seq = frames[i-SEQ_LEN:i]
            diffs = seq[1:] - seq[:-1]
            diffs = np.concatenate([np.zeros_like(diffs[:1]), diffs], axis=0)
            x = torch.tensor(np.stack([seq, diffs], axis=1),device = DEVICE).unsqueeze(0)

            with torch.no_grad():
                pred = model(x)[0, 0]
                gt = torch.tensor(frames[i], device=DEVICE)
                err = torch.mean((pred - gt) ** 2).item()

            fid = f"{int(vid)}_{frame_nums[i]}"
            score_map[fid] = err
            all_scores.append(err)

    # GLOBAL normalization 
    mn, mx = min(all_scores), max(all_scores)
    for k in score_map:
        score_map[k] = (score_map[k] - mn) / (mx - mn + 1e-8)

    # Fill into template — missing frames stay 0.0
    for i in range(len(sample)):
        if sample.at[i, "Id"] in score_map:
            sample.at[i, "Predicted"] = score_map[sample.at[i, "Id"]]

    sample.to_csv(SUBMISSION_PATH, index=False)
    print("submission.csv generated (no missing frames, zeros preserved)")
if __name__ == "__main__":
    train()
    



Using 2 GPU(s)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Epoch 1/7:   0%|          | 0/1139 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Epoch 1/7: 100%|██████████| 1139/1139 [

Epoch 1 | Loss: 0.004594


Epoch 2/7: 100%|██████████| 1139/1139 [07:36<00:00,  2.50it/s]


Epoch 2 | Loss: 0.002467
Saved model_epoch2.pth


Epoch 3/7: 100%|██████████| 1139/1139 [07:35<00:00,  2.50it/s]


Epoch 3 | Loss: 0.002259
Saved model_epoch3.pth


Epoch 4/7:   2%|▏         | 19/1139 [00:08<08:21,  2.23it/s]


KeyboardInterrupt: 

In [ ]:
# %% [code] {"execution":{"iopub.status.busy":"2025-12-30T05:05:15.895174Z","iopub.execute_input":"2025-12-30T05:05:15.895467Z","iopub.status.idle":"2025-12-30T05:05:17.957591Z","shell.execute_reply.started":"2025-12-30T05:05:15.895435Z","shell.execute_reply":"2025-12-30T05:05:17.956779Z"}}
import pandas as pd

# Load both submissions
ae = pd.read_csv("/kaggle/input/sample/submission-8.csv")
cl = pd.read_csv("/kaggle/input/cnn-lstm12/submission1-2.csv")

# Make sure rows align
assert (ae["Id"] == cl["Id"]).all()

# Weighted fusion
alpha = 0.95
ae["Predicted"] = alpha * cl["Predicted"] + (1 - alpha) * ae["Predicted"]

# Save fused submission
ae.to_csv("submission_fused2.csv", index=False)


In [2]:
import os, glob
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm


def extract_frame_number(p):
    return int(os.path.basename(p).split("_")[1].split(".")[0])


def generate_submission():
    # Load submission template
    sample = pd.read_csv(TEMPLATE_CSV)
    sample["Predicted"] = 0.0

    # Load trained RGB model
    model = CNNConvLSTMPredictor().to(DEVICE)
    state = torch.load("/kaggle/working/model_epoch2.pth", map_location=DEVICE)

    # Handle DataParallel checkpoints
    if any(k.startswith("module.") for k in state.keys()):
        state = {k.replace("module.", ""): v for k, v in state.items()}

    model.load_state_dict(state)
    model.eval()

    score_map = {}
    all_scores = []

    with torch.no_grad():
        for vid in tqdm(sorted(os.listdir(TEST_IMG_DIR)), desc="Vids"):
            paths = sorted(glob.glob(os.path.join(TEST_IMG_DIR, vid, "*.jpg")))

            # frames: (N, H, W, 3)
            frames = np.stack([
                unflip_if_needed(preprocess_frame(p)) for p in paths
            ])

            frame_nums = [extract_frame_number(p) for p in paths]

            for i in range(SEQ_LEN, len(frames)):
                seq = frames[i-SEQ_LEN:i]            # (T,H,W,3)

                diffs = np.concatenate(
                    [np.zeros_like(seq[:1]), seq[1:] - seq[:-1]],
                    axis=0
                )                                     # (T,H,W,3)

                # Move RGB into channel dimension
                seq   = np.transpose(seq,   (0, 3, 1, 2))   # (T,3,H,W)
                diffs = np.transpose(diffs, (0, 3, 1, 2))   # (T,3,H,W)

                x = torch.tensor(
                    np.concatenate([seq, diffs], axis=1),  # (T,6,H,W)
                    device=DEVICE
                ).unsqueeze(0)                              # (1,T,6,H,W)

                # Safety check
                assert x.dim() == 5, x.shape

                pred = model(x)[0, 0]                       # (H,W)
                gt = torch.tensor(frames[i], device=DEVICE).permute(2,0,1)

                err = torch.mean((pred - gt) ** 2).item()

                fid = f"{int(vid)}_{frame_nums[i]}"
                score_map[fid] = err
                all_scores.append(err)

    # -------- GLOBAL MIN–MAX NORMALIZATION ONLY --------
    all_scores = np.array(all_scores)
    mn, mx = all_scores.min(), all_scores.max()

    for k in score_map:
        score_map[k] = (score_map[k] - mn) / (mx - mn + 1e-8)

    # Fill template (missing frames stay 0)
    for i in range(len(sample)):
        fid = sample.at[i, "Id"]
        if fid in score_map:
            sample.at[i, "Predicted"] = score_map[fid]

    sample.to_csv(SUBMISSION_PATH, index=False)
    print("RGB submission.csv generated (global norm only)")

if __name__ == "__main__":
    # train()
    generate_submission()

Vids: 100%|██████████| 21/21 [09:33<00:00, 27.31s/it]


✅ RGB submission.csv generated (global norm only)
